In [2]:
!pip install -q ultralytics opencv-python onnx onnxruntime-gpu
import warnings
warnings.filterwarnings("ignore")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 8.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.

In [5]:
#CONFIG + AUTO PATH
import os
import cv2
import queue
import threading
import numpy as np
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
#wandb
# from kaggle_secrets import UserSecretsClient
# import wandb

# user_secrets = UserSecretsClient()
# key = user_secrets.get_secret("WANDB_API_KEY")

# wandb.login(key=key)
# wandb.init(
#     project="yolo-plate",
#     name="exp1"
# )

# wandb.config.update({
#     "imgsz": 512,
#     "batch": 16
# })

# ================= DEVICES =================
DEVICE_YOLO = "0,1"     # DDP YOLO
DEVICE_OCR = "cuda:1"    # PARSeq single GPU (optimal)
frame_q = queue.Queue(8)
det_q = queue.Queue(8)
# ================= AUTO DATA =================
DATA_ROOT = "/kaggle/input"


# def find_yaml():
#     for r, _, f in os.walk(DATA_ROOT):
#         for x in f:
#             if x.endswith(".yaml"):
#                 return os.path.join(r, x)
#     return None

DATA_YAML = "/kaggle/input/datasets/vdt1501/yaml-plate/data.yaml"
print("YOLO DATA YAML:", DATA_YAML)

# ❗ MANUAL OVERRIDE (nếu auto sai)
# DATA_YAML = "/kaggle/input/YOUR_DATASET/data.yaml"
# ===== QUEUES (PIPELINE BUFFER) =====


YOLO DATA YAML: /kaggle/input/datasets/vdt1501/yaml-plate/data.yaml


In [7]:
#YOLO TRAIN (TRANSFER + FASTEST)
from ultralytics import YOLO
import pandas as pd

model = YOLO("yolo26m.pt")

model.train(
    data="/kaggle/input/datasets/vdt1501/yaml-plate/data.yaml",

    epochs=100,

    imgsz=512,
    batch=24,

    device="0,1",   #dual GPU training

    workers=8,
    cache=True,

    amp=True,

    optimizer="AdamW",

    lr0=0.003,      # FAST transfer learning
    lrf=0.01,

    cos_lr=True,

    warmup_epochs=2,

    weight_decay=5e-4,

    patience=15,    # EARLY STOP

    close_mosaic=5,

    multi_scale=False,   

    project="/kaggle/working/yolo",
    name="exp",
    exist_ok=True,
    verbose=False,
    plots=False,
    # logger="wandb"
    # logger=None
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/datasets/vdt1501/yaml-plate/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, nam